# Student 360 in 1 Hour - Part 2: Context and Governance
⏱ **~20 min.**

**What you will do:** give the data business meaning with a **semantic view** and **Cortex Search**, then enforce who can see what with **roles, row access and masking**, and prove it by switching roles.

**Snowflake features you touch:** semantic views (metrics, synonyms, verified queries, custom instructions), Cortex Search (fuzzy match + RAG), row access policies, masking policies, the four-layer access model.

In [ ]:
%%sql -r set_context
USE ROLE STUDENT360_LAB_ADMIN;
USE DATABASE STUDENT360_LAB;
USE SCHEMA SEMANTIC;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT CURRENT_ROLE(), CURRENT_SCHEMA();

## 1. Search services and a course view
🛠️ `STUDENT_NAME_SEARCH` resolves typos in names to the exact value. `ADVISING_NOTES_SEARCH` gives semantic retrieval over the **redacted** notes.
📌 Use literal matching for identifier columns; use RAG for document text.

In [ ]:
%%sql -r create_course_view
CREATE OR REPLACE VIEW CURATED.COURSE_ENROLLMENT
  COMMENT = 'Course enrollments for students in the Student 360 (inherits its row-level security). DFW = final grade D/F/W, or midterm D/F in the current term.'
AS
SELECT
  e.ENROLLMENT_ID, e.STUDENT_ID, s.COLLEGE, e.TERM, e.COURSE_CODE, e.COURSE_TITLE, e.DEPARTMENT, e.CREDITS,
  e.FINAL_GRADE, e.MIDTERM_GRADE, e.GRADE_POINTS, e.ENROLLMENT_STATUS AS COURSE_STATUS,
  IFF(COALESCE(e.FINAL_GRADE, e.MIDTERM_GRADE) IN ('D','F','W'), 1, 0) AS DFW_FLAG
FROM RAW.SIS_ENROLLMENT e
JOIN CURATED.STUDENT_360 s ON s.STUDENT_ID = e.STUDENT_ID;
SELECT TERM, COUNT(*) AS ROWS_, ROUND(100 * AVG(DFW_FLAG), 1) AS DFW_PCT FROM CURATED.COURSE_ENROLLMENT GROUP BY 1 ORDER BY 1;

In [ ]:
%%sql -r create_name_search
CREATE OR REPLACE CORTEX SEARCH SERVICE AGENTS.STUDENT_NAME_SEARCH
  ON FULL_NAME
  WAREHOUSE = STUDENT360_LAB_WH
  TARGET_LAG = '1 hour'
  COMMENT = 'Fuzzy literal matching for student names, linked to the STUDENT_360_SV FULL_NAME dimension'
AS (SELECT DISTINCT FULL_NAME FROM CURATED.STUDENT_360);
SHOW CORTEX SEARCH SERVICES IN SCHEMA AGENTS;

In [ ]:
%%sql -r create_notes_search
CREATE OR REPLACE CORTEX SEARCH SERVICE AGENTS.ADVISING_NOTES_SEARCH
  ON NOTE_TEXT_REDACTED
  ATTRIBUTES STUDENT_ID, COLLEGE, NOTE_THEME, NOTE_TYPE, ADVISOR_NAME, NOTE_DATE
  WAREHOUSE = STUDENT360_LAB_WH
  TARGET_LAG = '1 hour'
  COMMENT = 'Semantic search over PII-redacted advising notes (Fall 2026)'
AS (
  SELECT NOTE_ID, NOTE_TEXT_REDACTED, STUDENT_ID, COLLEGE, NOTE_THEME, NOTE_TYPE, ADVISOR_NAME, NOTE_DATE
  FROM CURATED.ADVISING_NOTE_ENRICHED
);
SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH',
  '{"query": "student cannot afford tuition and may stop out", "columns": ["NOTE_TEXT_REDACTED","NOTE_THEME","COLLEGE"], "limit": 3}'
)):results AS TOP_3_NOTES;

## 2. The semantic view
🛠️ This is where business meaning is written once: what *DFW* and *HIGH risk* mean, synonyms like *at-risk students*, verified queries, and rules for rejecting out-of-bounds questions (contact details, adverse action).
📌 **Best practices:** descriptions are the #1 accuracy input; leave PII out of the model entirely; define metrics in the view, not in prompts.

In [ ]:
%%sql -r create_semantic_view
CREATE OR REPLACE SEMANTIC VIEW STUDENT360_LAB.SEMANTIC.STUDENT_360_SV

  TABLES (
    students AS STUDENT360_LAB.CURATED.STUDENT_360
      PRIMARY KEY (STUDENT_ID)
      WITH SYNONYMS = ('student', 'student 360', 'learner')
      COMMENT = 'One row per ACTIVE student at Lakeside University, unifying SIS, LMS, Financial Aid, Bursar, Advising CRM and Campus Card data, with an explainable retention-risk score. Snapshot as of week 10 of Fall 2026.',
    courses AS STUDENT360_LAB.CURATED.COURSE_ENROLLMENT
      PRIMARY KEY (ENROLLMENT_ID)
      WITH SYNONYMS = ('course enrollment', 'class', 'section')
      COMMENT = 'One row per student per course per term (Fall 2025, Spring 2026, Fall 2026). Fall 2026 is in progress: it has MIDTERM_GRADE but no FINAL_GRADE.',
    notes AS STUDENT360_LAB.CURATED.ADVISING_NOTE_ENRICHED
      PRIMARY KEY (NOTE_ID)
      WITH SYNONYMS = ('advising note', 'advisor note')
      COMMENT = 'One row per advising note from the Advising CRM. Text is PII-redacted. Each note is tagged with an AI-classified theme.'
  )

  RELATIONSHIPS (
    courses_to_students AS courses(STUDENT_ID) REFERENCES students,
    notes_to_students   AS notes(STUDENT_ID)   REFERENCES students
  )

  FACTS (
    students.CURRENT_GPA AS CURRENT_GPA
      COMMENT = 'Current grade point average on a 4.0 scale. Cumulative GPA for continuing students; midterm GPA for first-term freshmen.',
    students.RETENTION_RISK_SCORE AS RETENTION_RISK_SCORE
      COMMENT = 'Retention-risk score from 0 (no risk signals) to 100 (maximum). A triage signal to prioritise outreach, not a prediction about an individual.',
    students.UNMET_NEED AS UNMET_NEED
      COMMENT = 'Financial need in USD not covered by grants, scholarships or loans for 2026-27. Null if the student has no financial aid record.',
    students.BALANCE_DUE AS BALANCE_DUE
      COMMENT = 'Current outstanding Bursar balance in USD.',
    students.LOGIN_TREND_PCT AS LOGIN_TREND_PCT
      COMMENT = 'Change in average weekly LMS logins between weeks 1-3 and weeks 8-10. -0.5 means logins fell by half.',
    students.ASSIGNMENT_SUBMISSION_RATE AS ASSIGNMENT_SUBMISSION_RATE
      COMMENT = 'Share of LMS assignments due that were submitted this term (0 to 1).',
    students.CAMPUS_TOUCHPOINTS_PER_WEEK AS CAMPUS_TOUCHPOINTS_PER_WEEK
      COMMENT = 'Average weekly library visits + rec-center visits + events attended from campus ID-card swipes.',
    students.DAYS_SINCE_ADVISING AS DAYS_SINCE_ADVISING
      COMMENT = 'Days since the most recent advising note, as of 2 Nov 2026. 999 means no advising note on file.',
    students.CREDITS_EARNED AS CREDITS_EARNED
      COMMENT = 'Credit hours earned before the current term.',
    courses.CREDITS AS CREDITS
      COMMENT = 'Credit hours for the course.',
    courses.GRADE_POINTS AS GRADE_POINTS
      COMMENT = 'Grade points on a 4.0 scale (final grade; midterm grade for in-progress courses). Null for withdrawals.',
    courses.DFW_FLAG AS DFW_FLAG
      COMMENT = '1 if the course is a DFW: final grade D, F or W (or midterm D/F for in-progress courses); otherwise 0.'
  )

  DIMENSIONS (
    students.STUDENT_ID AS STUDENT_ID
      COMMENT = 'Unique student identifier from the Student Information System (format S0000123).',
    students.FULL_NAME AS FULL_NAME
      WITH SYNONYMS = ('student name', 'name')
      COMMENT = 'Student full name (first and last).'
      WITH CORTEX SEARCH SERVICE STUDENT360_LAB.AGENTS.STUDENT_NAME_SEARCH,
    students.COLLEGE AS COLLEGE
      WITH SYNONYMS = ('school', 'division')
      COMMENT = 'Academic college. Values: Engineering, Arts & Sciences, Business, Nursing & Health, Education.',
    students.MAJOR AS MAJOR
      WITH SYNONYMS = ('program', 'degree program')
      COMMENT = 'Declared major (for example Computer Science, Nursing, Accounting, Biology).',
    students.CLASS_LEVEL AS CLASS_LEVEL
      WITH SYNONYMS = ('year', 'class standing')
      COMMENT = 'Class standing: Freshman, Sophomore, Junior or Senior.',
    students.ADMIT_TERM AS ADMIT_TERM
      WITH SYNONYMS = ('cohort', 'entry term')
      COMMENT = 'Term the student was first admitted (their cohort), for example Fall 2024.',
    students.FIRST_GEN_FLAG AS FIRST_GEN_FLAG
      WITH SYNONYMS = ('first generation', 'first-gen')
      COMMENT = 'Y if the student is first-generation (first in family to attend college), otherwise N.',
    students.RESIDENCY AS RESIDENCY
      COMMENT = 'Residency classification: In-State, Out-of-State or International.',
    students.ASSIGNED_ADVISOR AS ASSIGNED_ADVISOR
      WITH SYNONYMS = ('advisor', 'academic advisor')
      COMMENT = 'Name of the student''s assigned academic advisor.',
    students.RISK_TIER AS RISK_TIER
      WITH SYNONYMS = ('risk level', 'at-risk status', 'retention risk')
      COMMENT = 'Retention-risk band derived from RETENTION_RISK_SCORE: HIGH (60 or more), MEDIUM (30 to 59), LOW (below 30).',
    students.TOP_RISK_DRIVERS AS TOP_RISK_DRIVERS
      WITH SYNONYMS = ('why at risk', 'risk factors', 'risk reasons')
      COMMENT = 'Semicolon-separated list of the rules that raised the risk score (for example Low GPA; Unmet financial need; Account hold).',
    students.HOLD_TYPE AS HOLD_TYPE
      WITH SYNONYMS = ('account hold', 'bursar hold')
      COMMENT = 'Bursar hold: NONE, FINANCIAL_HOLD or REGISTRATION_HOLD (blocks course registration).',
    students.SAP_STATUS AS SAP_STATUS
      WITH SYNONYMS = ('satisfactory academic progress')
      COMMENT = 'Financial-aid Satisfactory Academic Progress: MEETING, WARNING or SUSPENDED. Null if no aid record.',
    students.PELL_ELIGIBLE AS PELL_ELIGIBLE
      WITH SYNONYMS = ('pell grant', 'pell')
      COMMENT = 'Y if eligible for a federal Pell Grant (low-income indicator), otherwise N.',
    students.LATEST_CONCERN_THEME AS LATEST_CONCERN_THEME
      COMMENT = 'Theme of the student''s most recent concern-flagged advising note (financial hardship, mental health and wellbeing, academic struggle, considering leaving). Null if no concerns.',
    courses.TERM AS TERM
      COMMENT = 'Academic term: Fall 2025, Spring 2026 or Fall 2026 (current, in progress).',
    courses.COURSE_CODE AS COURSE_CODE
      COMMENT = 'Course code such as MATH101 or CHEM101.',
    courses.COURSE_TITLE AS COURSE_TITLE
      COMMENT = 'Course title such as Calculus I or General Chemistry.',
    courses.DEPARTMENT AS DEPARTMENT
      COMMENT = 'Academic department that owns the course.',
    courses.FINAL_GRADE AS FINAL_GRADE
      COMMENT = 'Final letter grade (A, B+, B, B-, C, D, F) or W for withdrawal. Null while the course is in progress.',
    courses.MIDTERM_GRADE AS MIDTERM_GRADE
      COMMENT = 'Midterm letter grade. Populated only for the in-progress Fall 2026 term.',
    courses.COURSE_STATUS AS COURSE_STATUS
      COMMENT = 'ENROLLED (in progress), COMPLETED or WITHDRAWN.',
    notes.NOTE_DATE AS NOTE_DATE
      COMMENT = 'Date the advising note was written.',
    notes.NOTE_TYPE AS NOTE_TYPE
      COMMENT = 'Advisor-selected category: Financial, Wellbeing, Academic, Retention, Career or Planning.',
    notes.NOTE_THEME AS NOTE_THEME
      COMMENT = 'AI-classified theme: financial hardship, mental health and wellbeing, academic struggle, considering leaving, or on track.',
    notes.ADVISOR_NAME AS ADVISOR_NAME
      COMMENT = 'Advisor who wrote the note.'
  )

  METRICS (
    students.STUDENT_COUNT AS COUNT(DISTINCT students.STUDENT_ID)
      WITH SYNONYMS = ('number of students', 'headcount', 'enrollment')
      COMMENT = 'Number of distinct active students.',
    students.AVG_GPA AS AVG(students.CURRENT_GPA)
      WITH SYNONYMS = ('average gpa', 'mean gpa')
      COMMENT = 'Average current GPA on a 4.0 scale.',
    students.AVG_RISK_SCORE AS AVG(students.RETENTION_RISK_SCORE)
      COMMENT = 'Average retention-risk score (0-100).',
    students.HIGH_RISK_STUDENT_COUNT AS COUNT(DISTINCT CASE WHEN students.RISK_TIER = 'HIGH' THEN students.STUDENT_ID END)
      WITH SYNONYMS = ('at-risk students', 'high risk students')
      COMMENT = 'Number of students in the HIGH retention-risk tier.',
    students.HIGH_RISK_PCT AS 100.0 * COUNT(DISTINCT CASE WHEN students.RISK_TIER = 'HIGH' THEN students.STUDENT_ID END) / NULLIF(COUNT(DISTINCT students.STUDENT_ID), 0)
      WITH SYNONYMS = ('at-risk rate', 'percent at risk')
      COMMENT = 'Percentage of students in the HIGH retention-risk tier.',
    students.TOTAL_UNMET_NEED AS SUM(students.UNMET_NEED)
      COMMENT = 'Total unmet financial need in USD across students.',
    students.TOTAL_BALANCE_DUE AS SUM(students.BALANCE_DUE)
      COMMENT = 'Total outstanding Bursar balance in USD.',
    students.STUDENTS_WITH_HOLDS AS COUNT(DISTINCT CASE WHEN students.HOLD_TYPE <> 'NONE' THEN students.STUDENT_ID END)
      COMMENT = 'Number of students with any Bursar hold.',
    students.AVG_LOGIN_TREND AS AVG(students.LOGIN_TREND_PCT)
      COMMENT = 'Average change in weekly LMS logins (early vs. recent weeks).',
    students.AVG_SUBMISSION_RATE AS AVG(students.ASSIGNMENT_SUBMISSION_RATE)
      COMMENT = 'Average LMS assignment submission rate (0-1).',
    courses.ENROLLMENT_COUNT AS COUNT(DISTINCT courses.ENROLLMENT_ID)
      COMMENT = 'Number of student-course enrollment records.',
    courses.DFW_COUNT AS SUM(courses.DFW_FLAG)
      COMMENT = 'Number of DFW course enrollments (D, F or W).',
    courses.DFW_RATE_PCT AS 100.0 * SUM(courses.DFW_FLAG) / NULLIF(COUNT(DISTINCT courses.ENROLLMENT_ID), 0)
      WITH SYNONYMS = ('dfw rate', 'failure rate', 'withdrawal rate')
      COMMENT = 'Percentage of course enrollments that are D, F or W.',
    notes.NOTE_COUNT AS COUNT(DISTINCT notes.NOTE_ID)
      COMMENT = 'Number of advising notes.',
    notes.CONCERN_NOTE_COUNT AS COUNT(DISTINCT CASE WHEN notes.CONCERN_FLAG THEN notes.NOTE_ID END)
      COMMENT = 'Number of advising notes whose AI-classified theme is a concern (anything other than on track).'
  )

  COMMENT = 'Lakeside University Student 360 - retention, academics, engagement, finances and advising for active students (Fall 2026, week 10).'

  AI_SQL_GENERATION '
    -- Business rules for SQL generation:
    -- 1. "At-risk" or "high risk" students means RISK_TIER = ''HIGH''. Unless asked otherwise rank by RETENTION_RISK_SCORE descending.
    -- 2. "Current term" means Fall 2026. Fall 2026 is in progress: use MIDTERM_GRADE, never FINAL_GRADE, for that term.
    -- 3. DFW rate = share of enrollments with a final grade D, F or W (or midterm D/F in progress). Use the DFW_RATE_PCT metric.
    -- 4. Always show STUDENT_ID next to FULL_NAME so duplicates and follow-ups are unambiguous.
    -- 5. When asked about an advisor''s caseload, filter on ASSIGNED_ADVISOR and order by RETENTION_RISK_SCORE descending.
    -- 6. Percentages are 0-100 (already multiplied). Round GPA to 2 decimals, money to whole dollars.
    -- 7. NULL UNMET_NEED means no financial-aid record; do not treat it as zero need.
    -- 8. Do not select or infer EMAIL, PHONE or DATE_OF_BIRTH. They are intentionally not in this model.
  '

  AI_QUESTION_CATEGORIZATION '
    -- Reject and explain:
    -- * Questions asking for contact details (email, phone, home address) or date of birth, or social security numbers: reply that contact details are not available through this assistant and to use the advising system of record.
    -- * Questions asking to rank or label individual students by race, ethnicity, religion, disability or immigration status.
    -- * Questions asking to decide admission, dismissal, aid revocation or other adverse action for a named student.
    -- Ask a clarifying question when:
    -- * A student name is ambiguous or matches more than one student: ask which STUDENT_ID.
    -- * The user says "last year" or "this year" without a term.
  '

  AI_VERIFIED_QUERIES (
    high_risk_by_college AS (
      QUESTION 'How many high-risk students do we have in each college?'
      SQL 'SELECT COLLEGE, COUNT(*) AS HIGH_RISK_STUDENTS, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS PCT_OF_HIGH_RISK
           FROM STUDENT360_LAB.CURATED.STUDENT_360 WHERE RISK_TIER = ''HIGH'' GROUP BY COLLEGE ORDER BY HIGH_RISK_STUDENTS DESC'
    ),
    top_risk_students AS (
      QUESTION 'Who are the 10 students we should reach out to first and why?'
      SQL 'SELECT STUDENT_ID, FULL_NAME, COLLEGE, MAJOR, RETENTION_RISK_SCORE, TOP_RISK_DRIVERS, ASSIGNED_ADVISOR
           FROM STUDENT360_LAB.CURATED.STUDENT_360 WHERE RISK_TIER = ''HIGH'' ORDER BY RETENTION_RISK_SCORE DESC, DAYS_SINCE_ADVISING DESC LIMIT 10'
    ),
    advisor_caseload_risk AS (
      QUESTION 'Show each advisor''s caseload size and how many of their students are high risk'
      SQL 'SELECT ASSIGNED_ADVISOR, COUNT(*) AS CASELOAD, COUNT_IF(RISK_TIER = ''HIGH'') AS HIGH_RISK, ROUND(AVG(RETENTION_RISK_SCORE), 1) AS AVG_RISK_SCORE
           FROM STUDENT360_LAB.CURATED.STUDENT_360 GROUP BY ASSIGNED_ADVISOR ORDER BY HIGH_RISK DESC'
    ),
    highest_dfw_courses AS (
      QUESTION 'Which courses have the highest DFW rate?'
      SQL 'SELECT COURSE_CODE, COURSE_TITLE, COUNT(*) AS ENROLLMENTS, ROUND(100.0 * SUM(DFW_FLAG) / COUNT(*), 1) AS DFW_RATE_PCT
           FROM STUDENT360_LAB.CURATED.COURSE_ENROLLMENT GROUP BY COURSE_CODE, COURSE_TITLE HAVING COUNT(*) >= 50 ORDER BY DFW_RATE_PCT DESC LIMIT 10'
    ),
    holds_and_risk AS (
      QUESTION 'Which students have a registration hold and are also high risk?'
      SQL 'SELECT STUDENT_ID, FULL_NAME, COLLEGE, BALANCE_DUE, DAYS_PAST_DUE, RETENTION_RISK_SCORE
           FROM STUDENT360_LAB.CURATED.STUDENT_360 WHERE HOLD_TYPE = ''REGISTRATION_HOLD'' AND RISK_TIER = ''HIGH'' ORDER BY BALANCE_DUE DESC'
    ),
    first_gen_vs_not AS (
      QUESTION 'Compare first-generation and continuing-generation students on GPA and risk'
      SQL 'SELECT FIRST_GEN_FLAG, COUNT(*) AS STUDENTS, ROUND(AVG(CURRENT_GPA), 2) AS AVG_GPA, ROUND(AVG(RETENTION_RISK_SCORE), 1) AS AVG_RISK_SCORE,
                  ROUND(100.0 * COUNT_IF(RISK_TIER = ''HIGH'') / COUNT(*), 1) AS HIGH_RISK_PCT
           FROM STUDENT360_LAB.CURATED.STUDENT_360 GROUP BY FIRST_GEN_FLAG'
    ),
    disengaged_not_advised AS (
      QUESTION 'Which students are disengaging in the LMS but have not seen an advisor in over 45 days?'
      SQL 'SELECT STUDENT_ID, FULL_NAME, COLLEGE, LOGIN_TREND_PCT, ASSIGNMENT_SUBMISSION_RATE, DAYS_SINCE_ADVISING, ASSIGNED_ADVISOR
           FROM STUDENT360_LAB.CURATED.STUDENT_360 WHERE LOGIN_TREND_PCT <= -0.4 AND DAYS_SINCE_ADVISING > 45 ORDER BY LOGIN_TREND_PCT ASC LIMIT 25'
    )
  );
SELECT 'Semantic view STUDENT_360_SV created' AS STATUS;

## 3. Validate before building any agent
🛠️ Query the view directly and reconcile to source. If the view is wrong, the agent will be wrong.

In [ ]:
%%sql -r test_sv_college
SELECT * FROM SEMANTIC_VIEW(
  STUDENT360_LAB.SEMANTIC.STUDENT_360_SV
  DIMENSIONS students.COLLEGE
  METRICS students.STUDENT_COUNT, students.HIGH_RISK_STUDENT_COUNT, students.HIGH_RISK_PCT, students.AVG_GPA
)
ORDER BY HIGH_RISK_PCT DESC;

In [ ]:
%%sql -r test_sv_reconcile
-- Same question answered straight from the table: the numbers must match the semantic-view result above.
SELECT COLLEGE, COUNT(*) AS STUDENT_COUNT, COUNT_IF(RISK_TIER = 'HIGH') AS HIGH_RISK_STUDENT_COUNT,
       ROUND(100.0 * COUNT_IF(RISK_TIER = 'HIGH') / COUNT(*), 2) AS HIGH_RISK_PCT, ROUND(AVG(CURRENT_GPA), 2) AS AVG_GPA
FROM STUDENT360_LAB.CURATED.STUDENT_360 GROUP BY COLLEGE ORDER BY HIGH_RISK_PCT DESC;

## 4. Roles and the four-layer access model
🛠️ Two personas: an **Engineering advisor** (their college's rows, PII visible) and an **IR analyst** (all rows, PII masked). Layers: platform, agent, tools, data. We grant three now; the agent layer comes in Part 3. Note there is **no grant on RAW**.

In [ ]:
%%sql -r four_layer_grants
USE ROLE ACCOUNTADMIN;
CREATE ROLE IF NOT EXISTS STUDENT360_LAB_ADVISOR_ENGINEERING COMMENT = 'Advisor (Engineering): CoWork-only consumer, Engineering rows, PII visible';
CREATE ROLE IF NOT EXISTS STUDENT360_LAB_IR_ANALYST  COMMENT = 'Institutional Research analyst: CoWork-only consumer, all rows, PII masked';
-- Let the lab admin (and ACCOUNTADMIN) switch into them to test
GRANT ROLE STUDENT360_LAB_ADVISOR_ENGINEERING TO ROLE STUDENT360_LAB_ADMIN;
GRANT ROLE STUDENT360_LAB_IR_ANALYST  TO ROLE STUDENT360_LAB_ADMIN;
GRANT ROLE STUDENT360_LAB_ADVISOR_ENGINEERING TO ROLE ACCOUNTADMIN;
GRANT ROLE STUDENT360_LAB_IR_ANALYST  TO ROLE ACCOUNTADMIN;

-- Layer 1: platform entitlement + compute
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE STUDENT360_LAB_IR_ANALYST;
GRANT USAGE ON WAREHOUSE STUDENT360_LAB_WH TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON WAREHOUSE STUDENT360_LAB_WH TO ROLE STUDENT360_LAB_IR_ANALYST;
GRANT USAGE ON DATABASE STUDENT360_LAB TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON DATABASE STUDENT360_LAB TO ROLE STUDENT360_LAB_IR_ANALYST;

-- Layer 3: tool / resource access (semantic view + search services)
GRANT USAGE ON SCHEMA STUDENT360_LAB.SEMANTIC TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON SCHEMA STUDENT360_LAB.SEMANTIC TO ROLE STUDENT360_LAB_IR_ANALYST;
GRANT USAGE ON SCHEMA STUDENT360_LAB.AGENTS   TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON SCHEMA STUDENT360_LAB.AGENTS   TO ROLE STUDENT360_LAB_IR_ANALYST;
GRANT SELECT ON SEMANTIC VIEW STUDENT360_LAB.SEMANTIC.STUDENT_360_SV TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT SELECT ON SEMANTIC VIEW STUDENT360_LAB.SEMANTIC.STUDENT_360_SV TO ROLE STUDENT360_LAB_IR_ANALYST;
GRANT USAGE ON CORTEX SEARCH SERVICE STUDENT360_LAB.AGENTS.STUDENT_NAME_SEARCH TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON CORTEX SEARCH SERVICE STUDENT360_LAB.AGENTS.STUDENT_NAME_SEARCH TO ROLE STUDENT360_LAB_IR_ANALYST;
-- (the notes search service is granted deliberately in Step 5, after the owner's-rights discussion)

-- Layer 4: data access on the CURATED layer only (policies still apply)
GRANT USAGE ON SCHEMA STUDENT360_LAB.CURATED TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT USAGE ON SCHEMA STUDENT360_LAB.CURATED TO ROLE STUDENT360_LAB_IR_ANALYST;
GRANT SELECT ON ALL TABLES         IN SCHEMA STUDENT360_LAB.CURATED TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT SELECT ON ALL TABLES         IN SCHEMA STUDENT360_LAB.CURATED TO ROLE STUDENT360_LAB_IR_ANALYST;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA STUDENT360_LAB.CURATED TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA STUDENT360_LAB.CURATED TO ROLE STUDENT360_LAB_IR_ANALYST;
GRANT SELECT ON ALL VIEWS          IN SCHEMA STUDENT360_LAB.CURATED TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT SELECT ON ALL VIEWS          IN SCHEMA STUDENT360_LAB.CURATED TO ROLE STUDENT360_LAB_IR_ANALYST;
-- Layer 2 (USAGE ON AGENT) is granted in Notebook 04.
SELECT 'Consumer roles created with layers 1, 3 and 4' AS STATUS;

## 5. Row-level and column-level policy
🛠️ One expression-based row policy scopes by college; masking policies hide email, phone and date of birth.
📌 A row policy containing a **subquery** disables change tracking and would break dynamic tables and search services, so this lab uses an expression policy.

In [ ]:
%%sql -r create_scope
USE ROLE STUDENT360_LAB_ADMIN;
USE SCHEMA GOVERNANCE;

CREATE OR REPLACE ROW ACCESS POLICY GOVERNANCE.COLLEGE_SCOPE_RAP AS (COLLEGE STRING) RETURNS BOOLEAN ->
  CURRENT_ROLE() IN ('STUDENT360_LAB_ADMIN', 'ACCOUNTADMIN', 'STUDENT360_LAB_IR_ANALYST')                       -- institution-wide roles
  OR CURRENT_ROLE() = 'STUDENT360_LAB_ADVISOR_' || UPPER(REGEXP_REPLACE(COLLEGE, '[^A-Za-z]', ''));  -- college-scoped advisors

ALTER DYNAMIC TABLE CURATED.STUDENT_360             ADD ROW ACCESS POLICY GOVERNANCE.COLLEGE_SCOPE_RAP ON (COLLEGE);
ALTER TABLE         CURATED.ADVISING_NOTE_ENRICHED  ADD ROW ACCESS POLICY GOVERNANCE.COLLEGE_SCOPE_RAP ON (COLLEGE);
ALTER VIEW          CURATED.COURSE_ENROLLMENT       ADD ROW ACCESS POLICY GOVERNANCE.COLLEGE_SCOPE_RAP ON (COLLEGE);
SELECT 'Row access policy applied to STUDENT_360, ADVISING_NOTE_ENRICHED, COURSE_ENROLLMENT' AS STATUS;

In [ ]:
%%sql -r create_masking
USE ROLE STUDENT360_LAB_ADMIN;
USE SCHEMA GOVERNANCE;

CREATE OR REPLACE MASKING POLICY GOVERNANCE.PII_STRING_MASK AS (val STRING) RETURNS STRING ->
  CASE WHEN CURRENT_ROLE() IN ('STUDENT360_LAB_ADMIN', 'ACCOUNTADMIN', 'STUDENT360_LAB_ADVISOR_ENGINEERING') THEN val
       ELSE '***MASKED***' END;

CREATE OR REPLACE MASKING POLICY GOVERNANCE.PII_DATE_MASK AS (val DATE) RETURNS DATE ->
  CASE WHEN CURRENT_ROLE() IN ('STUDENT360_LAB_ADMIN', 'ACCOUNTADMIN', 'STUDENT360_LAB_ADVISOR_ENGINEERING') THEN val
       ELSE NULL END;

ALTER DYNAMIC TABLE CURATED.STUDENT_360 MODIFY COLUMN EMAIL         SET MASKING POLICY GOVERNANCE.PII_STRING_MASK;
ALTER DYNAMIC TABLE CURATED.STUDENT_360 MODIFY COLUMN PHONE         SET MASKING POLICY GOVERNANCE.PII_STRING_MASK;
ALTER DYNAMIC TABLE CURATED.STUDENT_360 MODIFY COLUMN DATE_OF_BIRTH SET MASKING POLICY GOVERNANCE.PII_DATE_MASK;
SELECT 'Masking applied to EMAIL, PHONE, DATE_OF_BIRTH' AS STATUS;

## 6. Prove it: same query, different roles
📌 Persona tests use `USE SECONDARY ROLES NONE`. Otherwise your session keeps ACCOUNTADMIN privileges and the test proves nothing.

In [ ]:
%%sql -r test_as_advisor
USE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
-- Expect: only Engineering rows, real emails/phones
SELECT COLLEGE, COUNT(*) AS STUDENTS_VISIBLE FROM STUDENT360_LAB.CURATED.STUDENT_360 GROUP BY 1;
SELECT FULL_NAME, EMAIL, PHONE, DATE_OF_BIRTH, RISK_TIER FROM STUDENT360_LAB.CURATED.STUDENT_360 ORDER BY RETENTION_RISK_SCORE DESC LIMIT 3;
-- The semantic view and the course view inherit the same scope:
SELECT * FROM SEMANTIC_VIEW(STUDENT360_LAB.SEMANTIC.STUDENT_360_SV DIMENSIONS students.COLLEGE METRICS students.STUDENT_COUNT);
SELECT COLLEGE, COUNT(*) AS COURSE_ROWS_VISIBLE FROM STUDENT360_LAB.CURATED.COURSE_ENROLLMENT GROUP BY 1;

In [ ]:
%%sql -r test_as_ir
USE ROLE STUDENT360_LAB_IR_ANALYST;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
-- Expect: all colleges, but PII masked
SELECT COLLEGE, COUNT(*) AS STUDENTS_VISIBLE FROM STUDENT360_LAB.CURATED.STUDENT_360 GROUP BY 1 ORDER BY 1;
SELECT FULL_NAME, EMAIL, PHONE, DATE_OF_BIRTH, RISK_TIER FROM STUDENT360_LAB.CURATED.STUDENT_360 ORDER BY RETENTION_RISK_SCORE DESC LIMIT 3;

In [ ]:
%%sql -r test_no_raw
USE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
-- Expect an error: the consumer role has no privilege on the RAW schema (un-redacted notes live there)
SELECT NOTE_TEXT FROM STUDENT360_LAB.RAW.ADVISING_NOTE LIMIT 1;

## 7. The Cortex Search owner's-rights gotcha
⚠️ Search services run with the **owner's** rights. Row policies on the source table do not scope them. Run the next cell to see notes from other colleges leak, then fix it with a per-audience index.

In [ ]:
%%sql -r naive_grant
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON CORTEX SEARCH SERVICE STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
-- The advisor is scoped to Engineering by the row access policy ... but the search service is not
SELECT r.value:COLLEGE::STRING AS COLLEGE_RETURNED, COUNT(*) AS HITS
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH',
  '{"query": "student cannot afford tuition", "columns": ["COLLEGE"], "limit": 50}'))['results'])) r
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%%sql -r scoped_search
USE ROLE ACCOUNTADMIN;
REVOKE USAGE ON CORTEX SEARCH SERVICE STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH FROM ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
GRANT  USAGE ON CORTEX SEARCH SERVICE STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH TO ROLE STUDENT360_LAB_IR_ANALYST;   -- IR is institution-wide by design

USE ROLE STUDENT360_LAB_ADMIN;
USE WAREHOUSE STUDENT360_LAB_WH;
CREATE OR REPLACE CORTEX SEARCH SERVICE STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH_ENGINEERING
  ON NOTE_TEXT_REDACTED
  ATTRIBUTES STUDENT_ID, COLLEGE, NOTE_THEME, NOTE_TYPE, ADVISOR_NAME, NOTE_DATE
  WAREHOUSE = STUDENT360_LAB_WH
  TARGET_LAG = '1 hour'
  COMMENT = 'Engineering-only slice of the PII-redacted advising notes (scoped index; owner-rights safe)'
AS (
  SELECT NOTE_ID, NOTE_TEXT_REDACTED, STUDENT_ID, COLLEGE, NOTE_THEME, NOTE_TYPE, ADVISOR_NAME, NOTE_DATE
  FROM STUDENT360_LAB.CURATED.ADVISING_NOTE_ENRICHED
  WHERE COLLEGE = 'Engineering'
);
USE ROLE ACCOUNTADMIN;
GRANT USAGE ON CORTEX SEARCH SERVICE STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH_ENGINEERING TO ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
SELECT 'Engineering advisor now has only the Engineering-scoped notes index' AS STATUS;

In [ ]:
%%sql -r verify_scoped_search
USE ROLE STUDENT360_LAB_ADVISOR_ENGINEERING;
USE SECONDARY ROLES NONE;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT r.value:COLLEGE::STRING AS COLLEGE_RETURNED, COUNT(*) AS HITS
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH_ENGINEERING',
  '{"query": "student cannot afford tuition", "columns": ["COLLEGE"], "limit": 50}'))['results'])) r
GROUP BY 1;
-- And the shared service is now refused:
-- SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW('STUDENT360_LAB.AGENTS.ADVISING_NOTES_SEARCH', '{"query":"x","limit":1}');   -- expect: insufficient privileges

In [ ]:
%%sql -r reset_role
USE ROLE STUDENT360_LAB_ADMIN;
USE WAREHOUSE STUDENT360_LAB_WH;
SELECT CURRENT_ROLE() AS BACK_TO;

## ✅ Part 2 complete
A validated semantic view, two search services plus a scoped one, two persona roles, row and column policies, all proven by role switching.

**Takeaways:** write meaning once; least exposure; enforce access on the **data**, not the prompt; scope search indexes per audience.

**Next:** Part 3 builds the agents and puts them in CoWork.